In [ ]:
import numpy as np
import tensorflow as tf

In [ ]:
class PositionalIndex(tf.keras.layers.Layer):
    def call(self, x):
        bs = tf.shape(x)[0] # extract batch size
        number_of_vectors = tf.shape(x)[1] # how many vectors - we know it should be m*n, but let's count to be sure...
        indices = tf.range(number_of_vectors) # index for each vector
        indices = tf.expand_dims(indices, 0) # reshape it appropriately
        # tf.tile(indices, [bs, 1]): does the job of repeating the indices for each batch
        return tf.tile(indices, [bs, 1]) # repeat for each batch

In [ ]:
class ClassTokenIndex(tf.keras.layers.Layer):
    def call(self, x):
        bs = tf.shape(x)[0] # extract batch size
        number_of_vectors = 1 # how many vectors - we just want 1 (which is @ 0) ... we only want to generate 1 vector for the class token
        # now just get it to be the right size
        indices = tf.range(number_of_vectors) # index for each vector
        indices = tf.expand_dims(indices, 0) # reshape it appropriately
        return tf.tile(indices, [bs, 1]) # repeat for each batch

In [ ]:
def build_ViT(n,m,block_size,hidden_dim,num_layers,num_heads,key_dim,value_dim,mlp_dim,dropout_rate,num_classes):
    # n is number of rows of blocks
    # m is number of cols of blocks
    # query dimension = key dimension
    # block_size is number of pixels (with rgb) in each block
    inp = tf.keras.layers.Input(shape=(n*m,block_size))
    # hidden_dim: output dimension of the pink box
    mid = tf.keras.layers.Dense(hidden_dim, use_bias=False)(inp) # transform to vectors with different dimension
    # the positional embeddings
    inp2 = PositionalIndex()(inp)
    emb = tf.keras.layers.Embedding(input_dim=n*m, output_dim=hidden_dim)(inp2) # learned positional embedding for each of the n*m possible possitions
    mid = tf.keras.layers.Add()([mid, emb]) 
    # create and append class token to beginning of all input vectors
    tokenInd = ClassTokenIndex()(mid)
    token = tf.keras.layers.Embedding(input_dim=1, output_dim=hidden_dim)(tokenInd)
    mid = tf.keras.layers.Concatenate(axis=1)([token, mid])
    
    for l in range(num_layers): # how many Transformer Head layers are there?
        ln  = tf.keras.layers.LayerNormalization()(mid) # normalize
        mha = tf.keras.layers.MultiHeadAttention(num_heads=num_heads,key_dim=key_dim,value_dim=value_dim)(ln,ln,ln) # self attention!
        add = tf.keras.layers.Add()([mid,mha]) # add and norm
        ln  = tf.keras.layers.LayerNormalization()(add)
        den = tf.keras.layers.Dense(mlp_dim,activation='gelu')(ln) # maybe should be relu...who knows...
        den = tf.keras.layers.Dropout(dropout_rate)(den) # regularization
        den = tf.keras.layers.Dense(hidden_dim)(den) # back to the right dimensional space
        den = tf.keras.layers.Dropout(dropout_rate)(den)
        mid = tf.keras.layers.Add()([den,add]) # add and norm again
    
    fl = mid[:,0,:] # just grab the class token for each image in batch
    ln = tf.keras.layers.LayerNormalization()(fl)
    clas = tf.keras.layers.Dense(num_classes,activation='softmax')(ln) # probability that the image is in each category
    mod = tf.keras.models.Model(inp,clas)
    mod.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    return mod

In [ ]:
n = 7
m = 7
block_size = 16
hidden_dim = 32
num_layers = 4
num_heads = 4
key_dim = hidden_dim//num_heads # usually good practice for key_dim to be hidden_dim//num_heads...this is why we do Multi-Head attention
value_dim = key_dim*2
mlp_dim = hidden_dim
dropout_rate = 0.1
num_classes = 10



trans = build_ViT(n,m,block_size,hidden_dim,num_layers,num_heads,key_dim,value_dim,mlp_dim,dropout_rate,num_classes)
trans.summary()

In [ ]:
# 6,304 multi-head attention layer weights and biases
# query network, key network: hidden_dim*key_dim+key_dim
# value network: hidden_dim*value_dim+value_dim
# for num_heads heads: ((hidden_dim*key_dim+key_dim)*2 + hidden_dim*value_dim+value_dim)*num_heads
# output network: (value_dim*num_heads)*hidden_dim + hidden_dim
((hidden_dim*key_dim+key_dim)*2 + hidden_dim*value_dim+value_dim)*num_heads + (value_dim*num_heads)*hidden_dim + hidden_dim

In [ ]:
mnist = tf.keras.datasets.mnist

(x_train, y_train),(x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

ndata_train = x_train.shape[0]
ndata_test = x_test.shape[0]

In [ ]:
x_train.shape

In [ ]:
x_train_ravel = np.zeros((ndata_train,n*m,block_size))
for img in range(ndata_train):
    ind = 0
    for row in range(n):
        for col in range(m):
            x_train_ravel[img,ind,:] = x_train[img,(row*4):((row+1)*4),(col*4):((col+1)*4)].ravel()
            ind += 1
            

In [ ]:
x_test_ravel = np.zeros((ndata_test,n*m,block_size))
for img in range(ndata_test):
    ind = 0
    for row in range(n):
        for col in range(m):
            x_test_ravel[img,ind,:] = x_test[img,(row*4):((row+1)*4),(col*4):((col+1)*4)].ravel()
            ind += 1

In [ ]:
trans.fit(x_train_ravel,y_train,epochs=3,batch_size = 40,validation_split=0.15)

In [ ]:
out = trans.evaluate(x_test_ravel,y_test)

In [ ]:
out[1]